# Step 2b: Train COOLANT (Experimental Phased Pre-training)

**Experimental variant: train each module individually, then fine-tune jointly**

**Note**: The official COOLANT paper trains all 3 modules simultaneously per epoch.
This phased approach is architecturally motivated (Detection depends on CLIP features)
but differs from the paper. Compare against `2a_train_simultaneous.ipynb`.

## Phases
- Phase 1: CLIP pre-training (self-supervised, no labels needed)
- Phase 2: Similarity pre-training (contrastive pairs from real samples)
- Phase 3: Detection training (supervised, CLIP+Similarity frozen)
- Phase 4: Joint fine-tuning (all unfrozen, 0.1x LR)

Prerequisites: Run `0_verify_labels.ipynb` then `1_preprocess.ipynb` first.

Input: `../processed_data/hdf5/vifactcheck_*.h5`
Output: `../../training/checkpoints_phased/`

In [1]:
# Setup and imports
import sys
import os
import math
import random
import json
import warnings
import datetime
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import h5py
from tqdm import tqdm
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings("ignore")

# Project path (workflow/ -> notebooks/ -> project_root)
project_root = Path().absolute().parent.parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "src"))

print(f"Project root: {project_root}")

Project root: g:\My Drive\Thesis_Final\fake-new-detection


In [2]:
# Device and seeds
DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"🔧 Device: {DEVICE}")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# Paths
HDF5_DIR = project_root / "notebooks" / "processed_data" / "hdf5"
SAVE_DIR = project_root / "training" / "checkpoints_phased"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f"📂 HDF5 dir: {HDF5_DIR}")
print(f"💾 Save dir: {SAVE_DIR}")

🔧 Device: cuda
📂 HDF5 dir: g:\My Drive\Thesis_Final\fake-new-detection\notebooks\processed_data\hdf5
💾 Save dir: g:\My Drive\Thesis_Final\fake-new-detection\training\checkpoints_phased


In [3]:
# Configuration
CONFIG = {
    "batch_size": 32,
    "lr": 1e-3,
    "lr_clip": 1e-3,
    "weight_decay": 1e-5,
    "label_smoothing": 0.1,
    "grad_clip": 1.0,
    "max_epochs_phase1": 10,  # CLIP pre-training
    "max_epochs_phase2": 10,  # Similarity pre-training
    "max_epochs_phase3": 20,  # Detection training
    "max_epochs_phase4": 10,  # Joint fine-tuning
    "patience": 5,
    "dropout": 0.3,
}

print(f"🔧 Config: {CONFIG}")

🔧 Config: {'batch_size': 32, 'lr': 0.001, 'lr_clip': 0.001, 'weight_decay': 1e-05, 'label_smoothing': 0.1, 'grad_clip': 1.0, 'max_epochs_phase1': 10, 'max_epochs_phase2': 10, 'max_epochs_phase3': 20, 'max_epochs_phase4': 10, 'patience': 5, 'dropout': 0.3}


In [4]:
# Load data with weighted sampling for balanced batches
from src.processing.hdf5_dataset import create_hdf5_dataloaders_from_files

IMAGE_DIM = 2048
TEXT_EMBED = 768

train_path = HDF5_DIR / "vifactcheck_train.h5"
dev_path = HDF5_DIR / "vifactcheck_dev.h5"
test_path = HDF5_DIR / "vifactcheck_test.h5"

loaders, datasets = create_hdf5_dataloaders_from_files(
    train_path=str(train_path),
    dev_path=str(dev_path),
    test_path=str(test_path),
    batch_size=CONFIG["batch_size"],
    num_workers=0,
    weighted_sampling=True,  # Balance real/fake in each batch
)

# Verify label distribution
with h5py.File(str(train_path), "r") as f:
    train_labels = f["labels"][:]
label_dist = Counter(train_labels.tolist())
print(f"\nTraining label distribution: {dict(label_dist)}")
assert len(label_dist) >= 2, f"ERROR: Only {len(label_dist)} class in training data!"

# Compute class weights for CrossEntropyLoss
n_train = len(train_labels)
class_weight = torch.tensor(
    [n_train / (2 * label_dist.get(i, 1)) for i in range(2)],
    dtype=torch.float32,
).to(DEVICE)
print(f"Class weights: {class_weight.cpu().tolist()}")

print(f"\nData loaded: train={len(datasets['train'])}, dev={len(datasets['dev'])}, test={len(datasets['test'])}")

HDF5DatasetSimple: 3376 samples from vifactcheck_train.h5
HDF5DatasetSimple: 465 samples from vifactcheck_dev.h5
HDF5DatasetSimple: 973 samples from vifactcheck_test.h5
  Weighted sampling enabled: class_weights={1: 1.084484420173466, 0: 12.836501901140684}

DataLoaders created from separate files:
  Train: 106 batches (3376 samples)
  Dev:   15 batches (465 samples)
  Test:  31 batches (973 samples)

Training label distribution: {1: 3113, 0: 263}
Class weights: [6.418251037597656, 0.5422422289848328]

Data loaded: train=3376, dev=465, test=973


In [5]:
# Load model
from src.models.resnet_coolant import (
    ResNetCOOLANT,
    patch_encoding,
    patch_clip_projection,
    patch_cnn_with_dropout,
)

model = ResNetCOOLANT(CONFIG)

# Patch encodings for 2048-dim image features
patch_encoding(model.similarity_module.encoding, image_dim=IMAGE_DIM)
patch_encoding(model.detection_module.encoding, image_dim=IMAGE_DIM)
patch_encoding(model.detection_module.ambiguity_module.encoding, image_dim=IMAGE_DIM)

# Patch CLIP projections
patch_clip_projection(model.clip_module, target_dim=IMAGE_DIM, is_image=True)
patch_clip_projection(model.clip_module, target_dim=TEXT_EMBED, is_image=False)

# Patch FastCNN with dropout
patch_cnn_with_dropout(
    model.similarity_module.encoding.shared_text_encoding, TEXT_EMBED, CONFIG["dropout"]
)
patch_cnn_with_dropout(
    model.detection_module.encoding.shared_text_encoding, TEXT_EMBED, CONFIG["dropout"]
)
patch_cnn_with_dropout(
    model.detection_module.ambiguity_module.encoding.shared_text_encoding,
    TEXT_EMBED,
    CONFIG["dropout"],
)

model = model.to(DEVICE)
print(f"✅ Model loaded and patched")

✅ Model loaded and patched


In [6]:
# Loss functions with class weighting
# CLIP uses unweighted CE (self-supervised, targets are batch indices not class labels)
loss_ce_clip = nn.CrossEntropyLoss(label_smoothing=CONFIG["label_smoothing"])
# Detection uses class-weighted CE (supervised, class imbalance)
loss_ce_det = nn.CrossEntropyLoss(weight=class_weight, label_smoothing=CONFIG["label_smoothing"])
loss_kl = nn.KLDivLoss(reduction="batchmean")

print(f"Loss functions created (det class weights: {class_weight.cpu().tolist()})")

Loss functions created (det class weights: [6.418251037597656, 0.5422422289848328])


In [ ]:
# Phase 1: CLIP-only pre-training (self-supervised)
print("=" * 60)
print("PHASE 1: CLIP Pre-training (self-supervised, no labels needed)")
print("=" * 60)

# Freeze all modules except CLIP
for name, param in model.named_parameters():
    if "clip_module" not in name:
        param.requires_grad = False

optim_clip = torch.optim.AdamW(
    model.clip_module.parameters(), lr=CONFIG["lr_clip"], weight_decay=5e-4
)
sch_clip = torch.optim.lr_scheduler.CosineAnnealingLR(
    optim_clip, T_max=CONFIG["max_epochs_phase1"]
)

best_clip_loss = float("inf")

for epoch in range(1, CONFIG["max_epochs_phase1"] + 1):
    model.train()
    epoch_loss = 0.0
    n_samples = 0

    for text, image, _ in tqdm(loaders["train"], desc=f"Phase1 Epoch {epoch}"):
        text = text.to(DEVICE)
        image = image.to(DEVICE)
        bs = text.size(0)

        text_pooled = text.mean(dim=2)
        ie, te = model.clip_module(image, text_pooled)
        logits = ie @ te.T * math.exp(0.07)
        ids = torch.arange(bs, device=DEVICE)

        lc = (loss_ce_clip(logits, ids) + loss_ce_clip(logits.T, ids)) / 2

        optim_clip.zero_grad()
        lc.backward()
        torch.nn.utils.clip_grad_norm_(model.clip_module.parameters(), CONFIG["grad_clip"])
        optim_clip.step()

        epoch_loss += lc.item() * bs
        n_samples += bs

    epoch_loss /= n_samples
    sch_clip.step()

    # Validation
    model.eval()
    val_loss = 0.0
    val_n = 0
    with torch.no_grad():
        for text, image, _ in loaders["dev"]:
            text = text.to(DEVICE)
            image = image.to(DEVICE)
            bs = text.size(0)
            text_pooled = text.mean(dim=2)
            ie, te = model.clip_module(image, text_pooled)
            logits = ie @ te.T * math.exp(0.07)
            ids = torch.arange(bs, device=DEVICE)
            lc = (loss_ce_clip(logits, ids) + loss_ce_clip(logits.T, ids)) / 2
            val_loss += lc.item() * bs
            val_n += bs
    val_loss /= val_n

    print(f"  Epoch {epoch}: train_loss={epoch_loss:.4f}, val_loss={val_loss:.4f}")

    if val_loss < best_clip_loss:
        best_clip_loss = val_loss
        torch.save(model.clip_module.state_dict(), SAVE_DIR / "clip_phase1_best.pth")
        print(f"  >> Saved best CLIP checkpoint (val_loss={best_clip_loss:.4f})")

print(f"Phase 1 complete. Best CLIP val_loss: {best_clip_loss:.4f}")

PHASE 1: CLIP Pre-training (self-supervised, no labels needed)


Phase1 Epoch 1: 100%|██████████| 106/106 [10:06<00:00,  5.72s/it]


  Epoch 1: train_loss=3.4627, val_loss=3.4426
  >> Saved best CLIP checkpoint (val_loss=3.4426)


Phase1 Epoch 2: 100%|██████████| 106/106 [10:14<00:00,  5.80s/it]


  Epoch 2: train_loss=3.4625, val_loss=3.4426
  >> Saved best CLIP checkpoint (val_loss=3.4426)


...

Phase1 Epoch 8: 100%|██████████| 106/106 [10:03<00:00,  5.70s/it]


  Epoch 8: train_loss=3.4625, val_loss=3.4426


Phase1 Epoch 9: 100%|██████████| 106/106 [10:10<00:00,  5.76s/it]


  Epoch 9: train_loss=3.4625, val_loss=3.4426


Phase1 Epoch 10:  38%|███▊      | 40/106 [03:46<06:00,  5.46s/it]

In [ ]:
# Phase 2: Similarity pre-training with frozen CLIP
print("=" * 60)
print("PHASE 2: Similarity Pre-training (CLIP frozen)")
print("=" * 60)

# Freeze CLIP, unfreeze Similarity
for param in model.clip_module.parameters():
    param.requires_grad = False
for param in model.similarity_module.parameters():
    param.requires_grad = True
for param in model.detection_module.parameters():
    param.requires_grad = False

optim_sim = torch.optim.Adam(
    model.similarity_module.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"]
)
sch_sim = torch.optim.lr_scheduler.CosineAnnealingLR(
    optim_sim, T_max=CONFIG["max_epochs_phase2"]
)

loss_cos = nn.CosineEmbeddingLoss(margin=0.2)


def make_pair_sim(text, image, label, hard_negative_ratio=0.3):
    """Create paired/unpaired samples using only label=0 (real) samples."""
    batch_size = text.size(0)
    real_indices = (label == 0).nonzero(as_tuple=True)[0]
    used_fallback = len(real_indices) < 2
    if used_fallback:
        real_indices = torch.arange(batch_size, device=text.device)
    n_anchors = max(2, len(real_indices) // 2)
    anchor_indices = real_indices[torch.randperm(len(real_indices))[:n_anchors]]
    t_anchor = text[anchor_indices].clone()
    i_anchor = image[anchor_indices].clone()
    i_positive = i_anchor.clone()
    n_hard = int(n_anchors * hard_negative_ratio)
    n_random = n_anchors - n_hard
    i_negative = i_anchor.clone()
    if n_hard > 0:
        i_negative[:n_hard] = i_anchor[:n_hard].roll(1, dims=0)
    if n_random > 0:
        offset = random.randint(2, max(3, n_anchors // 2))
        i_negative[n_hard:] = i_anchor[n_hard:].roll(offset, dims=0)
    return t_anchor, i_positive, i_negative, used_fallback


best_sim_loss = float("inf")

for epoch in range(1, CONFIG["max_epochs_phase2"] + 1):
    model.train()
    epoch_loss = 0.0
    fallback_count = 0
    total_real_in_batches = 0
    n_batches = 0

    for text, image, label in tqdm(loaders["train"], desc=f"Phase2 Epoch {epoch}"):
        text = text.to(DEVICE)
        image = image.to(DEVICE)
        label = label.to(DEVICE)

        n_real = (label == 0).sum().item()
        total_real_in_batches += n_real

        ft, fi_m, fi_u, used_fb = make_pair_sim(text, image, label)
        if used_fb:
            fallback_count += 1

        ta_m, ia_m, _ = model.similarity_module(ft, fi_m)
        ta_u, ia_u, _ = model.similarity_module(ft, fi_u)
        t_cat = torch.cat([ta_m, ta_u])
        i_cat = torch.cat([ia_m, ia_u])
        y_cos = torch.cat([
            torch.ones(ta_m.size(0), device=DEVICE),
            -torch.ones(ta_u.size(0), device=DEVICE),
        ])
        ls = loss_cos(t_cat, i_cat, y_cos)

        optim_sim.zero_grad()
        ls.backward()
        torch.nn.utils.clip_grad_norm_(model.similarity_module.parameters(), CONFIG["grad_clip"])
        optim_sim.step()

        epoch_loss += ls.item()
        n_batches += 1

    epoch_loss /= n_batches
    avg_real = total_real_in_batches / n_batches
    sch_sim.step()

    print(f"  Epoch {epoch}: loss={epoch_loss:.4f}, avg_real/batch={avg_real:.1f}, fallbacks={fallback_count}/{n_batches}")

    if epoch_loss < best_sim_loss:
        best_sim_loss = epoch_loss
        torch.save(model.similarity_module.state_dict(), SAVE_DIR / "similarity_phase2_best.pth")
        print(f"  >> Saved best Similarity checkpoint")

print(f"Phase 2 complete. Best sim_loss: {best_sim_loss:.4f}")

In [ ]:
# Phase 3: Detection training with frozen CLIP + Similarity
print("=" * 60)
print("PHASE 3: Detection Training (CLIP + Similarity frozen)")
print("=" * 60)

# Freeze CLIP and Similarity, unfreeze Detection
for param in model.clip_module.parameters():
    param.requires_grad = False
for param in model.similarity_module.parameters():
    param.requires_grad = False
for param in model.detection_module.parameters():
    param.requires_grad = True

optim_det = torch.optim.Adam(
    model.detection_module.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"]
)
sch_det = torch.optim.lr_scheduler.CosineAnnealingLR(
    optim_det, T_max=CONFIG["max_epochs_phase3"]
)

best_val_f1 = 0.0
patience_counter = 0

for epoch in range(1, CONFIG["max_epochs_phase3"] + 1):
    model.train()
    epoch_loss = 0.0
    all_y, all_p = [], []
    total = 0

    for text, image, label in tqdm(loaders["train"], desc=f"Phase3 Epoch {epoch}"):
        text = text.to(DEVICE)
        image = image.to(DEVICE)
        label = label.to(DEVICE)

        text_pooled = text.mean(dim=2)
        with torch.no_grad():
            ie, te = model.clip_module(image, text_pooled)

        det, attn, skl = model.detection_module(text, image, te, ie)
        ld = loss_ce_det(det, label) + 0.5 * loss_kl(
            F.log_softmax(attn, 1), F.softmax(skl, 1)
        )

        optim_det.zero_grad()
        ld.backward()
        torch.nn.utils.clip_grad_norm_(model.detection_module.parameters(), CONFIG["grad_clip"])
        optim_det.step()

        pred = det.argmax(1)
        all_y.extend(label.cpu().numpy())
        all_p.extend(pred.cpu().numpy())
        total += label.size(0)
        epoch_loss += ld.item() * label.size(0)

    epoch_loss /= total
    train_f1 = f1_score(all_y, all_p, average="macro", zero_division=0)
    train_acc = accuracy_score(all_y, all_p)
    sch_det.step()

    # Validation with per-class metrics
    model.eval()
    val_loss = 0.0
    val_y, val_p = [], []
    val_total = 0
    with torch.no_grad():
        for text, image, label in loaders["dev"]:
            text = text.to(DEVICE)
            image = image.to(DEVICE)
            label = label.to(DEVICE)
            text_pooled = text.mean(dim=2)
            ie, te = model.clip_module(image, text_pooled)
            det, attn, skl = model.detection_module(text, image, te, ie)
            ld = loss_ce_det(det, label) + 0.5 * loss_kl(
                F.log_softmax(attn, 1), F.softmax(skl, 1)
            )
            pred = det.argmax(1)
            val_y.extend(label.cpu().numpy())
            val_p.extend(pred.cpu().numpy())
            val_total += label.size(0)
            val_loss += ld.item() * label.size(0)

    val_loss /= val_total
    val_f1 = f1_score(val_y, val_p, average="macro", zero_division=0)
    val_acc = accuracy_score(val_y, val_p)
    prec, rec, _, _ = precision_recall_fscore_support(val_y, val_p, average=None, zero_division=0)
    cm = confusion_matrix(val_y, val_p)

    print(f"  Epoch {epoch}: train_loss={epoch_loss:.4f} train_F1={train_f1:.4f}")
    print(f"    val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_F1={val_f1:.4f}")
    print(f"    real: prec={prec[0]:.3f} rec={rec[0]:.3f} | fake: prec={prec[1]:.3f} rec={rec[1]:.3f}")
    print(f"    CM: {cm.tolist()}")

    # Early stopping on macro F1
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        torch.save(model.detection_module.state_dict(), SAVE_DIR / "detection_phase3_best.pth")
        print(f"  >> New best macro_F1={best_val_f1:.4f}, saved checkpoint")
    else:
        patience_counter += 1
        if patience_counter >= CONFIG["patience"]:
            print(f"\n  Early stopping at epoch {epoch}")
            break

print(f"Phase 3 complete. Best val macro_F1: {best_val_f1:.4f}")

In [ ]:
# Phase 4: Joint fine-tuning (unfreeze all, reduce LR)
print("=" * 60)
print("PHASE 4: Joint Fine-tuning (all modules unfrozen, 0.1x LR)")
print("=" * 60)

# Unfreeze all
for param in model.parameters():
    param.requires_grad = True

optim_joint = torch.optim.Adam(
    model.parameters(), lr=CONFIG["lr"] * 0.1, weight_decay=CONFIG["weight_decay"]
)
sch_joint = torch.optim.lr_scheduler.CosineAnnealingLR(
    optim_joint, T_max=CONFIG["max_epochs_phase4"]
)

loss_cos = nn.CosineEmbeddingLoss(margin=0.2)

best_joint_f1 = 0.0
patience_counter = 0

for epoch in range(1, CONFIG["max_epochs_phase4"] + 1):
    model.train()
    sim_loss_total = 0.0
    clip_loss_total = 0.0
    det_loss_total = 0.0
    all_y, all_p = [], []
    total = 0
    n_batches = 0

    for text, image, label in tqdm(loaders["train"], desc=f"Phase4 Epoch {epoch}"):
        text = text.to(DEVICE)
        image = image.to(DEVICE)
        label = label.to(DEVICE)
        bs = label.size(0)

        # Similarity
        ft, fi_m, fi_u, _ = make_pair_sim(text, image, label)
        ta_m, ia_m, _ = model.similarity_module(ft, fi_m)
        ta_u, ia_u, _ = model.similarity_module(ft, fi_u)
        t_cat = torch.cat([ta_m, ta_u])
        i_cat = torch.cat([ia_m, ia_u])
        y_cos = torch.cat([
            torch.ones(ta_m.size(0), device=DEVICE),
            -torch.ones(ta_u.size(0), device=DEVICE),
        ])
        ls = loss_cos(t_cat, i_cat, y_cos)
        sim_loss_total += ls.item()

        # CLIP
        text_pooled = text.mean(dim=2)
        ie, te = model.clip_module(image, text_pooled)
        logits = ie @ te.T * math.exp(0.07)
        ids = torch.arange(bs, device=DEVICE)
        ts, is_, _ = model.similarity_module(text, image)
        soft_m = is_ @ ts.T * math.exp(0.07)
        lc = (loss_ce_clip(logits, ids) + loss_ce_clip(logits.T, ids)) / 2
        ls2 = (
            -(F.softmax(soft_m, 1) * F.log_softmax(logits, 1)).sum(1).mean()
            + -(F.softmax(soft_m.T, 1) * F.log_softmax(logits.T, 1)).sum(1).mean()
        ) / 2
        l_clip = lc + 0.2 * ls2
        clip_loss_total += l_clip.item()

        # Detection (uses class-weighted loss)
        det, attn, skl = model.detection_module(text, image, te, ie)
        ld = loss_ce_det(det, label) + 0.5 * loss_kl(
            F.log_softmax(attn, 1), F.softmax(skl, 1)
        )
        det_loss_total += ld.item() * bs

        pred = det.argmax(1)
        all_y.extend(label.cpu().numpy())
        all_p.extend(pred.cpu().numpy())
        total += bs
        n_batches += 1

        # Joint loss
        total_loss = ls + l_clip + ld

        optim_joint.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
        optim_joint.step()

    train_f1 = f1_score(all_y, all_p, average="macro", zero_division=0)
    train_acc = accuracy_score(all_y, all_p)
    sch_joint.step()

    # Validation with per-class metrics
    model.eval()
    val_y, val_p = [], []
    val_total = 0
    with torch.no_grad():
        for text, image, label in loaders["dev"]:
            text = text.to(DEVICE)
            image = image.to(DEVICE)
            label = label.to(DEVICE)
            text_pooled = text.mean(dim=2)
            ie, te = model.clip_module(image, text_pooled)
            det, attn, skl = model.detection_module(text, image, te, ie)
            pred = det.argmax(1)
            val_y.extend(label.cpu().numpy())
            val_p.extend(pred.cpu().numpy())
            val_total += label.size(0)

    val_f1 = f1_score(val_y, val_p, average="macro", zero_division=0)
    val_acc = accuracy_score(val_y, val_p)
    prec, rec, _, _ = precision_recall_fscore_support(val_y, val_p, average=None, zero_division=0)
    cm = confusion_matrix(val_y, val_p)

    print(f"  Epoch {epoch}: train_acc={train_acc:.4f} train_F1={train_f1:.4f}")
    print(f"    val_acc={val_acc:.4f} val_F1={val_f1:.4f}")
    print(f"    real: prec={prec[0]:.3f} rec={rec[0]:.3f} | fake: prec={prec[1]:.3f} rec={rec[1]:.3f}")
    print(f"    CM: {cm.tolist()}")
    print(f"    losses: sim={sim_loss_total/n_batches:.4f} clip={clip_loss_total/n_batches:.4f} det={det_loss_total/total:.4f}")

    # Early stopping on macro F1
    if val_f1 > best_joint_f1:
        best_joint_f1 = val_f1
        patience_counter = 0
        torch.save(model.state_dict(), SAVE_DIR / "full_model_phase4_best.pth")
        print(f"  >> New best macro_F1={best_joint_f1:.4f}, saved full model")
    else:
        patience_counter += 1
        if patience_counter >= CONFIG["patience"]:
            print(f"\n  Early stopping at epoch {epoch}")
            break

print(f"Phase 4 complete. Best joint macro_F1: {best_joint_f1:.4f}")
print(f"\nFinal summary:")
print(f"  Phase 1 (CLIP):       best val_loss = {best_clip_loss:.4f}")
print(f"  Phase 2 (Similarity): best loss     = {best_sim_loss:.4f}")
print(f"  Phase 3 (Detection):  best macro_F1 = {best_val_f1:.4f}")
print(f"  Phase 4 (Joint):      best macro_F1 = {best_joint_f1:.4f}")